# EX_04 — Chatbots básicos (ejercicios)

**Notebook de referencia:** `notebook/04_Chatbots_Basicos.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Lista de mensajes

Construye `messages = [{"role": "system", ...}, {"role": "user", ...}]` para un asistente de estudio que **nunca** da la solución directa sino pistas.


In [1]:
messages = [
    {
        "role": "system",
        "content": (
            "Eres un asistente de estudio paciente, empático y puramente educativo. "
            "Tu regla de oro absoluta es que JAMÁS debes dar la solución directa o la respuesta "
            "final a ningún problema, ecuación o pregunta que te haga el estudiante. "
            "En su lugar, debes guiarlo paso a paso utilizando el método socrático: haz preguntas "
            "orientadoras, señala conceptos clave o proporciona pistas sutiles. "
            "Si el estudiante te pide desesperadamente la respuesta directa, niégate amablemente "
            "y ofrécele una pista aún más pequeña para ayudarle a resolverlo por sí mismo."
        )
    },
    {
        "role": "user",
        "content": "¿Me puedes decir ya el resultado de 15 * 24? Estoy atascado y necesito terminar estos deberes rápido."
    }
]

# Comprobación de la estructura imprimiendo los roles cargados
print(f"Lista de mensajes creada con éxito. Contiene {len(messages)} elementos.")
print(f"Primer rol: '{messages[0]['role']}' | Segundo rol: '{messages[1]['role']}'")


Lista de mensajes creada con éxito. Contiene 2 elementos.
Primer rol: 'system' | Segundo rol: 'user'


## Actividad 2 — Historial acotado

Implementa una función `trim_history(messages, max_turns)` que conserve system + los últimos N intercambios user/assistant.


In [2]:
from typing import Any

def trim_history(messages: list[dict[str, Any]], max_turns: int) -> list[dict[str, Any]]:
    # 1. Extraer el mensaje del sistema (si existe) para protegerlo
    system_message = [msg for msg in messages if msg.get("role") == "system"]

    # 2. Filtrar todos los mensajes de la conversación que no sean de sistema (user y assistant)
    chat_messages = [msg for msg in messages if msg.get("role") != "system"]

    # 3. Calcular cuántos mensajes individuales representan los 'max_turns'
    # Como un turno completo es un par (User + Assistant), multiplicamos por 2
    messages_to_keep = max_turns * 2

    # 4. Tomar los últimos N mensajes usando rebanado negativo (slicing)
    trimmed_chat = chat_messages[-messages_to_keep:] if messages_to_keep > 0 else []

    # 5. Retornar la combinación del mensaje de sistema original con el historial recortado
    return system_message + trimmed_chat


# =====================================================================
# PRUEBA DE VERIFICACIÓN (Para que compruebes que funciona en tu celda)
# =====================================================================
historial_de_prueba = [
    {"role": "system", "content": "Eres un tutor."},
    {"role": "user", "content": "Hola (Turno 1)"},
    {"role": "assistant", "content": "¡Hola! ¿En qué te ayudo? (Turno 1)"},
    {"role": "user", "content": "Tengo una duda de mates (Turno 2)"},
    {"role": "assistant", "content": "Dime, soy todo oídos (Turno 2)"},
    {"role": "user", "content": "Frase nueva del usuario (Turno 3)"}
]

# Recortamos para quedarnos únicamente con los últimos 2 turnos
historial_acotado = trim_history(historial_de_prueba, max_turns=2)

print("--- COMPROBACIÓN DE HISTORIAL ---")
print(f"Mensajes originales: {len(historial_de_prueba)}")
print(f"Mensajes tras trim_history (debe incluir system + 4 últimos): {len(historial_acotado)}\n")
print("Historial resultante:")
for m in historial_acotado:
    print(f" -> {m['role']}: {m['content']}")

--- COMPROBACIÓN DE HISTORIAL ---
Mensajes originales: 6
Mensajes tras trim_history (debe incluir system + 4 últimos): 5

Historial resultante:
 -> system: Eres un tutor.
 -> assistant: ¡Hola! ¿En qué te ayudo? (Turno 1)
 -> user: Tengo una duda de mates (Turno 2)
 -> assistant: Dime, soy todo oídos (Turno 2)
 -> user: Frase nueva del usuario (Turno 3)


In [3]:
# 1. Definimos el prompt que le daremos al LLM para realizar la compresión
summarize_prompt = """Eres un asistente experto en gestión de memoria.
Se te va a proporcionar un historial de chat largo entre un usuario y un asistente.

Tu tarea es generar un resumen ejecutivo, claro y compacto (en un máximo de 3 o 4 líneas) que condense los puntos clave discutidos, las decisiones tomadas y las dudas resueltas hasta el momento.

Este resumen se utilizará como contexto para los próximos mensajes, por lo que es vital que mantengas los datos más importantes y elimines los saludos o el relleno.

Historial a resumir:
{chat_history_text}

Resumen de la conversación:"""


# 2. Lógica de control en Python para medir el volumen de palabras
def check_and_summarize(messages: list[dict], max_words: int = 800) -> bool:
    """
    Simula el disparador que detecta si el hilo de conversación ha superado
    el límite de palabras para lanzar la petición de resumen al LLM.
    """
    # Unimos todo el texto actual del chat (excluyendo el prompt de sistema)
    full_chat_text = " ".join([msg["content"] for msg in messages if msg["role"] != "system"])

    # Contamos las palabras utilizando el método split() sugerido en el enunciado
    word_count = len(full_chat_text.split())

    print(f"Conteo actual: {word_count} palabras en el historial de chat.")

    # Si supera el umbral, devolvemos True indicando que hay que resumir
    if word_count > max_words:
        print(f"¡Alerta! Se han superado las {max_words} palabras. Disparando 'summarize_prompt'...")
        return True

    print("El historial está dentro de los límites seguros. No hace falta resumir aún.")
    return False


# =====================================================================
# SIMULACIÓN PRÁCTICA DEL COMPORTAMIENTO
# =====================================================================
print("--- PROMPT DE RESUMEN CARGADO ---")
print(summarize_prompt[:134] + "...\n")

# Simulamos un historial corto (no debería dispararse)
chat_corto = [
    {"role": "user", "content": "Hola, necesito ayuda con un script de python para limpiar datos."},
    {"role": "assistant", "content": "¡Hola! Claro que sí, muéstrame el código y lo revisamos juntos."}
]
check_and_summarize(chat_corto, max_words=800)

--- PROMPT DE RESUMEN CARGADO ---
Eres un asistente experto en gestión de memoria. 
Se te va a proporcionar un historial de chat largo entre un usuario y un asistente.
...

Conteo actual: 22 palabras en el historial de chat.
El historial está dentro de los límites seguros. No hace falta resumir aún.


False